# 🚀 HELIO YAJNA — Free High-Speed GPU Inference Server
### Run on Google Colab (Tesla T4 GPU) or Kaggle (P100 / T4 GPU)
This notebook turns a free Colab or Kaggle instance into an ultra-fast GPU API for the Helio Yajna platform.

**Speed comparison:**
- Local CPU: ~2,500ms per scan
- Intel Iris Xe: ~750ms per scan
- **Google Colab / Kaggle NVIDIA T4 GPU: ~25ms per scan (100x faster!)**

### Step 1: Check GPU Acceleration
Ensure your Runtime is set to GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies
Installs Ultralytics YOLO, FastAPI, Uvicorn, and Cloudflared / Localtunnel for public tunneling.

In [ ]:
!pip install -q ultralytics fastapi uvicorn python-multipart
!pip install -q pyngrok nest_asyncio

### Step 3: Download Model Weights (weights.pt)
Upload your `weights.pt` file to the Colab files sidebar, or download it directly.

In [ ]:
import os
if not os.path.exists('weights.pt'):
    print('Please upload weights.pt to this notebook directory, or provide a download link.')
else:
    print('weights.pt found and ready!')

### Step 4: Start the GPU Inference API Server

In [ ]:
# Write server script
code = '''
import os, base64, time, cv2, numpy as np, torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from ultralytics import YOLO

app = FastAPI(title='Helio GPU Engine')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = YOLO('weights.pt').to(DEVICE)

class PredictRequest(BaseModel):
    image_base64: str
    lat: float
    lon: float
    buffer_radius_sqft: Optional[int] = 2400
    initial_conf: Optional[float] = 0.15
    fallback_conf: Optional[float] = 0.05

def calc_m_per_px(lat, zoom=20):
    import math
    return (156543.03392 * math.cos(math.radians(lat))) / (2 ** zoom)

def calc_radius_px(area_sqft, m_px):
    import math
    return math.sqrt(area_sqft * 0.092903 / math.pi) / m_px

def calc_overlap(box, center, r):
    bx1, by1, bx2, by2 = box
    cx, cy = center
    ix1 = max(bx1, cx - r); iy1 = max(by1, cy - r)
    ix2 = min(bx2, cx + r); iy2 = min(by2, cy + r)
    return (ix2 - ix1) * (iy2 - iy1) if (ix2 > ix1 and iy2 > iy1) else 0.0

@app.get('/')
def health():
    return {'status': 'online', 'device': DEVICE, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}

@app.post('/predict')
def predict(req: PredictRequest):
    t0 = time.time()
    img_bytes = base64.b64decode(req.image_base64)
    img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    m_px = calc_m_per_px(req.lat, 20)
    r1200 = calc_radius_px(1200, m_px)
    r2400 = calc_radius_px(2400, m_px)

    res = model.predict(img, device=DEVICE, conf=req.initial_conf, imgsz=1024, half=torch.cuda.is_available(), verbose=False)[0]
    boxes = res.boxes.xyxy.tolist() if res.boxes else []
    confs = res.boxes.conf.tolist() if res.boxes else []

    has_solar = False; best_box = []; best_conf = 0.0; buf = 2400; method = 'not_found'
    best_idx = -1; max_ov = 0.0
    for i, b in enumerate(boxes):
        ov = calc_overlap(b, center, r1200)
        if ov > max_ov: max_ov = ov; best_idx = i
    if best_idx != -1:
        has_solar = True; best_box = boxes[best_idx]; best_conf = confs[best_idx]; buf = 1200; method = 'gpu_t4_initial'
    else:
        for i, b in enumerate(boxes):
            ov = calc_overlap(b, center, r2400)
            if ov > max_ov: max_ov = ov; best_idx = i
        if best_idx != -1:
            has_solar = True; best_box = boxes[best_idx]; best_conf = confs[best_idx]; buf = 2400; method = 'gpu_t4_2400'

    pv_area = 0.0; dist_m = 0.0
    if has_solar and best_box:
        pv_area = (best_box[2] - best_box[0]) * (best_box[3] - best_box[1]) * (m_px ** 2)
        cx_b = (best_box[0] + best_box[2]) / 2.0
        cy_b = (best_box[1] + best_box[3]) / 2.0
        dist_m = np.sqrt((cx_b - center[0])**2 + (cy_b - center[1])**2) * m_px

    return {
        'has_solar': has_solar,
        'confidence': round(float(best_conf), 4),
        'bbox': best_box,
        'buffer_size': buf,
        'pv_area_sqm': round(float(pv_area), 2),
        'euclidean_distance': round(float(dist_m), 2),
        'detection_method': method,
        'device': DEVICE,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'inference_latency_ms': round((time.time() - t0) * 1000.0, 1)
    }
'''
with open('gpu_app.py', 'w') as f:
    f.write(code)
print('gpu_app.py created!')

### Step 5: Launch Server & Expose via Cloudflare Tunnel (Free, No Auth Token Needed)
Run this cell. It downloads cloudflared, starts Uvicorn on port 8000, and prints your public GPU API URL.

In [ ]:
import subprocess, time
# Start uvicorn in background
proc = subprocess.Popen(['uvicorn', 'gpu_app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(3)

# Install and run cloudflared tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

print('=======================================================================')
print('STARTING PUBLIC GPU TUNNEL...')
print('Look for the trycloudflare.com URL below:')
print('Copy that URL into your Helio .env as: GPU_INFERENCE_API_URL=https://...')
print('=======================================================================')

!cloudflared tunnel --url http://localhost:8000